## Plate Solving Simulation

A simplified simulation of how plate-solving works. A fake star catalog is generated, a rotated camera field of view is cropped from it, and a 4-star **quad** is extracted and normalized (translation, rotation, and scale removed) to serve as a sky fingerprint. The matched quad is then located back in the full catalog — mirroring how real solvers like Astrometry.net identify where an image is pointing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform

In [ ]:
# 1. Generate fake sky
np.random.seed(1)

N_STARS = 200
sky = np.random.uniform(0,100,(N_STARS,2))

In [ ]:
# 2. Simulate camera image

center = np.array([50,60])
radius = 10

mask = np.linalg.norm(sky-center,axis=1) < radius
camera_stars = sky[mask]

theta = np.deg2rad(30)

R = np.array([
    [np.cos(theta),-np.sin(theta)],
    [np.sin(theta), np.cos(theta)]
])

camera_image = (camera_stars-center) @ R.T

In [ ]:
# 3. Pick quad
quad_idx = np.random.choice(len(camera_image),4,replace=False)
quad = camera_image[quad_idx]

In [ ]:
# 4. Normalize quad properly
D = squareform(pdist(quad))
i, j = np.unravel_index(np.argmax(D), D.shape)

A = quad[i]
B = quad[j]

baseline = B - A
baseline_len = np.linalg.norm(baseline)

# Step 1: translate
translated = quad - A

# Step 2: rotate so baseline aligns with x-axis
theta = np.arctan2(baseline[1], baseline[0])

R = np.array([
    [np.cos(-theta), -np.sin(-theta)],
    [np.sin(-theta),  np.cos(-theta)]
])

rotated = translated @ R.T

# Step 3: scale
quad_norm = rotated / baseline_len

In [ ]:
# 5. Visualization (four separate figures)
# SKY
plt.figure(figsize=(6,5), dpi=100)
plt.scatter(sky[:,0],sky[:,1],s=5)
plt.title("Full Sky")
ax = plt.gca()
ax.set_aspect("equal")
ax.set_box_aspect(1)

# CAMERA IMAGE
plt.figure(figsize=(6,5), dpi=100)
plt.scatter(camera_image[:,0],camera_image[:,1],s=20)
plt.scatter(quad[:,0],quad[:,1],c="red",s=60)
plt.title("Camera Image\n(quad chosen)")
ax = plt.gca()
ax.set_aspect("equal")
ax.set_box_aspect(1)

# NORMALIZED QUAD
plt.figure(figsize=(6,5), dpi=100)
plt.scatter(quad_norm[:,0], quad_norm[:,1], c="red", s=120)
ax = plt.gca()

# draw quad edges
for k in range(4):
    for m in range(k+1,4):
        plt.plot(
            [quad_norm[k,0], quad_norm[m,0]],
            [quad_norm[k,1], quad_norm[m,1]],
            color="black", alpha=0.4
        )

# draw baseline
plt.plot([0,1],[0,0],color="black",linewidth=3)

plt.title("Normalized Quad\n(rotation + scale removed)")
ax.set_aspect("equal")
ax.set_box_aspect(1)
plt.xlim(-0.5,1.5)
plt.ylim(-1,1)

# MATCHED QUAD IN SKY
plt.figure(figsize=(6,5), dpi=100)
plt.scatter(sky[:,0],sky[:,1],s=5)
match = sky[mask][quad_idx]
plt.scatter(match[:,0],match[:,1],c="red",s=80)
plt.title("Same Quad in Sky")
ax = plt.gca()
ax.set_aspect("equal")
ax.set_box_aspect(1)

plt.show()